# ChargeSquare Analytics — A1–A6

`energy_kwh` is the **cumulative** per-session meter register — the classic **energy trap**.
Every energy metric below uses per-session deltas, **never** `SUM(energy_kwh)`, which would
over-count by ~(readings per session).

This notebook runs the six analytical queries in `analytics/queries/` against ClickHouse
(over its HTTP interface, port 8123) and writes each result to `analytics/output/*.csv`.


In [1]:
import os
from pathlib import Path
import clickhouse_connect
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Resolve the pipeline root robustly: honour $PIPELINE_ROOT if set, otherwise walk
# upward from the current directory until we find the one that holds BOTH
# docker-compose.yml and analytics/queries. Works from the repo root, from analytics/,
# or from any nested subdirectory.
def find_root():
    env = os.environ.get("PIPELINE_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "docker-compose.yml").exists() and (d / "analytics/queries").is_dir():
            return d
    raise RuntimeError(
        "could not locate pipeline root (need docker-compose.yml + analytics/queries); "
        "set PIPELINE_ROOT to override"
    )

ROOT = find_root()
QUERIES = ROOT / "analytics/queries"
OUTPUT = ROOT / "analytics/output"
OUTPUT.mkdir(parents=True, exist_ok=True)

client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username="chargesquare", password="chargesquare", database="ev",
)

LABELS = ["A1", "A2", "A3", "A4", "A5", "A6"]

def run(label):
    # clickhouse-connect wants a single statement, so strip the trailing ';'.
    sql = sorted(QUERIES.glob(f"{label}_*.sql"))[0].read_text(encoding="utf-8").strip().rstrip(";")
    df = client.query_df(sql)
    df.to_csv(OUTPUT / f"{label}.csv", index=False)
    print(f"{label}: {len(df)} rows -> analytics/output/{label}.csv")
    return df

def save_plot(name):
    try:
        plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()
    except Exception as e:
        print("plot skipped:", e)
    finally:
        plt.close("all")


## A1 — Hourly energy consumption (per-session deltas, not cumulative sums)


In [2]:
a1 = run("A1")
try:
    a1.plot(x="hour", y="energy_kwh", figsize=(10,3), legend=False, title="A1 Hourly energy (kWh)")
    save_plot("A1")
except Exception as e:
    print("plot skipped:", e)
a1.head()


A1: 5 rows -> analytics/output/A1.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,hour,energy_kwh
0,2026-07-01 14:00:00,285111.650
1,2026-07-01 15:00:00,688626.317
2,2026-07-01 16:00:00,673172.832
3,2026-07-01 17:00:00,662881.084
4,2026-07-01 18:00:00,416805.608


## A2 — Station uptime / downtime, worst stations per operator


In [3]:
a2 = run("A2")
try:
    top = a2.head(12).iloc[::-1]
    top.plot(kind="barh", x="station_id", y="downtime_s", figsize=(10,4), legend=False, title="A2 Downtime seconds (top stations)")
    save_plot("A2")
except Exception as e:
    print("plot skipped:", e)
a2.head(12)


A2: 20 rows -> analytics/output/A2.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,station_id,total_s,downtime_s,uptime_ratio
0,ZES,TR-IST-1000,60924,7142,0.8828
1,Trugo,TR-IZM-0074,61424,5919,0.9036
2,VoltRun,TR-BUR-0250,60924,5916,0.9029
3,ZES,TR-KON-0067,60924,5629,0.9076
4,Trugo,TR-IZM-0705,45693,5306,0.8839
5,Esarj,TR-IST-1877,60924,5110,0.9161
6,VoltRun,TR-BUR-0085,45693,5070,0.8890
7,ChargeSquare,TR-IST-1732,60924,4823,0.9208
8,ZES,TR-IZM-0468,60924,4691,0.9230
9,ChargeSquare,TR-IZM-0514,30462,4675,0.8465


## A3 — Average duration & energy by vehicle brand


In [4]:
a3 = run("A3")
try:
    a3.plot(kind="bar", x="brand", y=["avg_duration_min","avg_energy_kwh"], figsize=(10,3), title="A3 Avg duration (min) & energy (kWh) by brand")
    save_plot("A3")
except Exception as e:
    print("plot skipped:", e)
a3


A3: 9 rows -> analytics/output/A3.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,brand,sessions,avg_duration_min,avg_energy_kwh
0,Tesla,32407,28.4,32.27
1,Volvo,11449,31.4,32.02
2,Volkswagen,6710,33.4,32.85
3,Hyundai,6067,30.1,35.17
4,Togg,4486,32.6,36.58
5,Kia,4406,29.9,34.79
6,Renault,3755,31.5,28.44
7,BMW,3722,31.0,35.53
8,Mercedes-Benz,2947,34.8,28.74


## A4 — Revenue and peak-hour contribution, per operator


In [5]:
a4 = run("A4")
try:
    a4.plot(kind="bar", x="operator_id", y=["revenue_eur","peak_revenue_eur"], figsize=(10,3), title="A4 Revenue (EUR): total vs peak-hour")
    save_plot("A4")
except Exception as e:
    print("plot skipped:", e)
a4


A4: 140 rows -> analytics/output/A4.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,city,tariff_id,revenue_eur,peak_revenue_eur,peak_pct,sessions
0,VoltRun,Istanbul,standard-v1,35094.76,10334.71,29.4,2713
1,ChargeSquare,Istanbul,standard-v1,33454.43,9992.71,29.9,2591
2,Esarj,Istanbul,standard-v1,32748.42,9756.73,29.8,2562
3,Trugo,Istanbul,standard-v1,32264.28,9360.85,29.0,2576
4,ZES,Istanbul,standard-v1,30927.99,9388.23,30.4,2473
...,...,...,...,...,...,...,...
135,ChargeSquare,Konya,fleet-v1,1170.76,373.04,31.9,106
136,VoltRun,Konya,fleet-v1,1156.21,360.02,31.1,125
137,VoltRun,Konya,off-peak-v1,1109.07,278.21,25.1,144
138,ZES,Konya,off-peak-v1,1000.93,186.62,18.6,105


## A5 — Geographic distribution of FAULT events (deduped by event_id)


In [6]:
a5 = run("A5")
try:
    a5.plot(kind="bar", x="city", y="fault_count", figsize=(10,3), legend=False, title="A5 Fault count by city")
    save_plot("A5")
except Exception as e:
    print("plot skipped:", e)
a5


A5: 7 rows -> analytics/output/A5.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,city,fault_count,stations_affected,faults_per_station,lat,lon
0,Istanbul,1098,838,1.31,41.0077,28.9767
1,Ankara,477,349,1.37,39.9352,32.8604
2,Izmir,370,285,1.30,38.4236,27.1434
3,Bursa,231,182,1.27,40.1909,29.0625
4,Antalya,222,168,1.32,36.8960,30.7162
5,Adana,143,109,1.31,36.9936,35.3180
6,Konya,98,74,1.32,37.8700,32.4901


## A6 — Anomaly detection: sessions > 2 sigma above the fleet mean power


In [7]:
a6 = run("A6")
try:
    a6.head(20).plot(kind="bar", x="session_id", y="z_score", figsize=(10,3), legend=False, title="A6 Anomalous sessions (z-score)")
    save_plot("A6")
except Exception as e:
    print("plot skipped:", e)
a6.head(20)


A6: 100 rows -> analytics/output/A6.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_40596/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,session_id,station_id,brand,avg_power_kw,z_score
0,sess-8b5a2d2e-4fb5-4842,TR-ANK-0163,Tesla,250.0,2.41
1,sess-bb1534d6-ae87-411d,TR-ANK-0389,Tesla,250.0,2.41
2,sess-587cd780-3100-4ce4,TR-IST-0827,Tesla,250.0,2.41
3,sess-f907bbd9-096e-44f3,TR-IST-0245,Tesla,250.0,2.41
4,sess-4fcddaab-d9e2-4f12,TR-IZM-0393,Tesla,250.0,2.41
5,sess-dc0b2a86-47f5-47fd,TR-IST-1444,Tesla,250.0,2.41
6,sess-7f7e2f6e-8ebc-4fe4,TR-ANT-0044,Tesla,250.0,2.41
7,sess-5c804329-a731-47be,TR-IST-0370,Tesla,250.0,2.41
8,sess-d64dbd88-bd46-49d3,TR-IST-0981,Tesla,250.0,2.41
9,sess-d20ef5cc-b4a8-40c7,TR-ANK-0633,Tesla,250.0,2.41


## Summary

All six results are in `analytics/output/*.csv`. The energy figures (A1, A3) use
per-session **deltas** of the cumulative `energy_kwh` register — summing the raw column
would over-count by roughly the number of meter readings per session.
